# DriftMind Benchmark Suite: Evaluation vs. Adaptive ARIMA
This notebook serves as a rigorous performance evaluation of DriftMind, an advanced system for real-time signal monitoring and forecasting, compared against a highly adaptive implementation of the Auto-Regressive Integrated Moving Average (ARIMA) model.

## What is DriftMind?
DriftMind is a specialized tool designed to solve the challenges of monitoring high-frequency, non-stationary signals. It functions as a forecasting and change-detection engine that addresses "Concept Drift"—the phenomenon where a signal's underlying statistical properties change over time.

### Key capabilities include:

- Real-time Adaptation: It continuously updates its internal parameters to match the current regime of a signal without requiring manual intervention.

- Multi-Domain Flexibility: It is engineered to handle signals across diverse sectors, including Network Telemetry (TCP/UDP), Industrial Sensors (IoT), and Financial time-series.

- Point-of-Change Detection: Beyond forecasting, it is capable of identifying the exact moment a signal drifts, making it vital for anomaly detection and predictive maintenance.

## Scientific Integrity & Latency Considerations
To ensure a mathematically sound and fair comparison, the following constraints apply:

On-Premises Latency Simulation: DriftMind core libraries are proprietary and currently accessible freely via a REST API. To avoid unfair latency comparisons between a cloud-hosted API and a local library (like ARIMA), the DriftMind results used here reflect on-premises execution speeds. These represent the performance of the paid, local binary version rather than a network-delayed API call.

Genuine Results: All DriftMind predictions provided in the dataset are genuine, verifiable, and generated by the core algorithm, despite being processed in a local offline environment for benchmarking purposes.

## Benchmarking Scope & Methodology
Current Focus: Each notebook in this suite compares DriftMind against one specific technology. This particular notebook evaluates performance against Adaptive ARIMA using a reactive retraining logic.

Data Dimensionality: At this stage, all datasets included are univariate. While Driftmind natively supports multivariate signals, benchmarks are currently under development.

Benchmark Angle: The ARIMA model used here is "Triggered," meaning it utilizes drift-detection metrics (MASE and Correlation) to simulate a competitor that is as "smart" and adaptable as DriftMind.

## An "Alive" Open-Source Benchmark
This suite is designed as a living project. We encourage community participation to increase the robustness of our evaluations:

- New Datasets: We are actively adding new datasets. If you have unique univariate or multivariate signals that challenge forecasting models, we welcome your contributions.

- New Baselines: We accept proposals for new benchmark models (e.g., Prophet, Transformer-based models, or custom DSP filters).

### Requirements for Inclusion: Any proposed baseline must be:

-  Cold-start ready: Capable of initializing with minimal or none historical data.

- Adaptable: Capable of handling non-stationary signals.

- Verifiable: Code and results must be fully transparent and reproducible.

## 1. Environment Setup and Dependency Loading

This section prepares the workspace for the benchmarking experiment. It handles system paths to ensure local modules are accessible and imports the necessary libraries for data manipulation, visualization, and time-series modeling.

#### Key Components:
* **Data Science Stack**: `pandas` and `numpy` for efficient data handling.
* **Visualization**: `matplotlib` and `seaborn` for generating high-fidelity performance charts.
* **Path Management**: Utilizing `pathlib` and `sys.path.insert` to allow the notebook to import custom baseline models from the project's `/src` directory.
* **Path Constants**: `ANALYSIS_CSV` and `EXPERIMENTS_DIR` define the data layout in a single place — update them here if the benchmark data is relocated.
* **Benchmark Target**: Imports the **`TriggeredARIMABaseline`**, which represents the **Reactive Angle**—an ARIMA model that adapts its parameters only when statistical drift is detected.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

# --- Dynamic Path Configuration ---
# This ensures that the custom 'src' directory is in the Python path,
# allowing us to import baselines regardless of where the notebook is run.
sys.path.insert(0, str(Path.cwd().parent / "src"))

# --- Baseline Import ---
# Loading the 'Triggered' variant specifically for this experiment.
# This class separates state updates (fast) from parameter re-fitting (slow).
from arima.t_arima import TriggeredARIMABaseline

In [ ]:
# --- Path Constants ---
# Single source of truth for the data layout.
# Update these if the benchmark data is moved to a different location.
_DATA_ROOT = Path("data") / "driftmind_results"
ANALYSIS_CSV = _DATA_ROOT / "analysis_results.csv"
EXPERIMENTS_DIR = _DATA_ROOT / "experiments"

## 2. Exploratory Data Analysis (EDA) and Global Performance Metrics

This section loads the primary analysis log to provide a "bird's-eye view" of DriftMind's performance across all processed datasets. By aggregating results into categories, we can identify architectural strengths and weaknesses before diving into individual baseline comparisons.

#### Analytical Objectives:
1. **Metadata Loading**: Ingest the `analysis_results.csv` containing performance logs (MAE, sMAE, Throughput) for every experiment.
2. **Categorical Profiling**: Group data by 'Category' to calculate average precision ($sMAE$) and efficiency ($Throughput$).
3. **Statistical Visualization**: 
    * **Precision Check**: Bar charts comparing the normalized error across different data regimes.
    * **Efficiency Check**: Visualizing processing speed (TPS) to identify computational bottlenecks.
    * **Robustness Check**: A scatter plot analyzing the relationship between signal volatility ($\sigma$) and model error, identifying if higher variance leads to linear degradation in accuracy.

In [ ]:
# --- DATA INGESTION ---
# Load the master results file. Using 'pd.read_csv' allows us to verify
# the integrity of all previous DriftMind runs.
df = pd.read_csv(ANALYSIS_CSV)

# Configuration to ensure high-visibility when auditing the raw table in Jupyter
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Display raw experiment log
display(df)

# --- SECTION 1: Categorical Metrics Aggregation ---
# We group by 'Category' (e.g., Network, Sensor, Financial) to see
# how DriftMind generalizes across different domains.
summary = (
    df.groupby("Category")
    .agg(
        {
            "Dataset": "count",
            "sMAE": "mean",  # Accuracy metric (Normalized)
            "Throughput": "mean",  # Efficiency metric
            "Points": "sum",  # Volume handled
        }
    )
    .rename(
        columns={
            "Dataset": "File_Count",
            "sMAE": "Avg_sMAE",
            "Throughput": "Avg_TPS",
            "Points": "Total_Points",
        }
    )
    .sort_values("Avg_sMAE")
)

print("### Summary Metrics per Category ###")
display(summary)

# --- SECTION 2: Visualization Suite ---
sns.set_theme(style="whitegrid")

# Graph 1: Comparative Precision
# Lower sMAE indicates higher precision. This identifies 'easy' vs 'hard' categories.
plt.figure(figsize=(10, 6))
sns.barplot(
    x=summary.index,
    y="Avg_sMAE",
    data=summary,
    palette="viridis",
    hue=summary.index,
    legend=False,
)
plt.title("Average $sMAE$ by Category (Lower is Better)")
plt.ylabel("Normalized Mean Absolute Error ($sMAE$)")
plt.xlabel("Data Category")
plt.xticks(rotation=45)
plt.show()

# Graph 2: Comparative Efficiency
# High TPS is critical for real-time streaming applications.
plt.figure(figsize=(10, 6))
sns.barplot(
    x=summary.index,
    y="Avg_TPS",
    data=summary,
    palette="magma",
    hue=summary.index,
    legend=False,
)
plt.title("Average Throughput by Category")
plt.ylabel("Predictions per Second")
plt.xlabel("Data Category")
plt.xticks(rotation=45)
plt.show()

# Graph 3: Error Sensitivity (Volatility vs. Error)
# This scatter plot investigates if the model is sensitive to data noise.
# We use a Log Scale for StdDev to handle disparate data magnitudes.
plt.figure(figsize=(10, 6))
sns.scatterplot(x="StdDev", y="sMAE", hue="Category", data=df, s=100, alpha=0.7)
plt.xscale("log")
plt.title(r"Model Robustness: $\sigma$ vs. $sMAE$")
plt.xlabel(r"Standard Deviation ($\sigma$) - Log Scale")
plt.ylabel("Normalized Error ($sMAE$)")
plt.grid(True, which="both", ls="--")
plt.show()

## 3. Individual Experiment Deep-Dive: Metadata & Visualization

The `visualize_experiment_full` function is designed to provide a comprehensive look at a single experiment's lifecycle. It automates the lookup of statistical summaries and generates synchronized time-series plots.

#### Functional Workflow:
1. **Metadata Retrieval**: Searches the master analysis file to display the specific attributes (Mean, StdDev, Min/Max) for the target ID.
2. **Automated File Matching**: Uses regex-style pattern matching to locate the correct experimental CSV even when filenames are long or complex.
3. **Dual-Pane Visualization**:
    * **Primary Axis**: Overlays the Ground Truth (Actual) with the DriftMind Prediction (Expected) to visualize tracking accuracy.
    * **Error Axis**: Plots the Absolute Error (AE) to highlight specific timestamps where the model struggled.
4. **Warm-up Identification**: Automatically detects and highlights the "Warm-up" phase (where the model is learning the initial signal parameters) using gray shading.

In [ ]:
def visualize_experiment_full(
    experiment_id,
    main_csv=None,
    base_path=None,
):
    """
    Retrieves metadata and generates a performance dashboard for a specific experiment.

    Args:
        experiment_id (int): The unique ID assigned to the data run.
        main_csv (Path | None): Path to the summarized results table.
            Defaults to ANALYSIS_CSV.
        base_path (Path | None): Directory containing raw prediction logs.
            Defaults to EXPERIMENTS_DIR.
    """
    if main_csv is None:
        main_csv = ANALYSIS_CSV
    if base_path is None:
        base_path = EXPERIMENTS_DIR

    # 1. Load and search Metadata from main CSV
    try:
        df_main = pd.read_csv(main_csv)
    except Exception as e:
        print(f"Error loading {main_csv}: {e}")
        return

    # Filter for the specific experiment row
    exp_info = df_main[df_main["Experiment Id"] == experiment_id]

    if exp_info.empty:
        print(f"Error: Experiment ID {experiment_id} not found in {main_csv}")
        return

    # --- Print Technical Summary ---
    # This provides the context of the data (e.g., its scale and range)
    print(f"{'=' * 60}")
    print(f" METADATA FOR EXPERIMENT {experiment_id}")
    print(f"{'=' * 60}")
    for col in exp_info.columns:
        print(f"{col:<15}: {exp_info.iloc[0][col]}")
    print(f"{'=' * 60}\n")

    # 2. Dynamic File Search
    # Uses glob to handle filenames with variable timestamps or descriptive suffixes
    files = list(base_path.glob(f"Experiment_{experiment_id}_*.csv"))

    if not files:
        print(f"Error: No result file found for ID {experiment_id} in {base_path}")
        return

    file_path = files[0]
    file_name = file_path.name

    # 3. Load Prediction Streaming Data
    # skipinitialspace=True ensures clean column names if CSV has leading spaces
    df_results = pd.read_csv(file_path, skipinitialspace=True)

    # 4. Create Synchronized Visualization
    # Height ratios are 3:1 to prioritize the signal view over the error view
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(15, 10), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
    )

    ax1.set_title(f"File: {file_name}", fontsize=14, fontweight="bold", pad=20)

    # --- Top Plot: Actual vs. Forecast ---
    ax1.plot(
        df_results.index,
        df_results["Actual"],
        label="Actual",
        color="#1f77b4",
        alpha=0.8,
    )
    ax1.plot(
        df_results.index,
        df_results["Expected"],
        label="Expected (DriftMind)",
        color="#ff7f0e",
        linestyle="--",
        linewidth=2,
    )

    ax1.set_ylabel("Value")
    ax1.legend(loc="best")
    ax1.grid(True, alpha=0.3)

    # --- Bottom Plot: Residual Error ---
    # fill_between provides a quick visual 'area' check for error magnitude
    ax2.fill_between(df_results.index, df_results["AE"], color="red", alpha=0.15)
    ax2.plot(
        df_results.index, df_results["AE"], color="red", linewidth=1, label="Abs Error"
    )

    ax2.set_ylabel("AE")
    ax2.set_xlabel("Time Steps")
    ax2.grid(True, alpha=0.3)

    # --- Warm-up Period Logic ---
    # Identifies rows where the model was not yet outputting predictions
    warmup_count = df_results["Expected"].isna().sum()
    if warmup_count > 0:
        ax1.axvspan(0, warmup_count, color="gray", alpha=0.1, label="Warm-up")
        ax2.axvspan(0, warmup_count, color="gray", alpha=0.1)

    plt.tight_layout()
    plt.show()

## 4. Experiment Execution: Single-Run Analysis

This cell triggers the `visualize_experiment_full` function for a specific `experiment_id`. It is used to validate the streaming performance of **DriftMind** on a case-by-case basis.

#### Objectives:
1. **Target Identification**: Specify the ID of the experiment to be audited.
2. **Visual Inspection**: Manually verify how closely the "Expected" line tracks the "Actual" signal.
3. **Error Analysis**: Identify if the **Absolute Error (AE)** is consistent or if it spikes during specific signal transitions (e.g., sudden level shifts or frequency changes).
4. **Metadata Verification**: Review the printed metadata table to ensure the data range and standard deviation align with the visual scale of the plot.

In [ ]:
# --- EXECUTION BLOCK ---
# We call the visualization function for Experiment 1.
# This ID can be changed to any 'Experiment Id' found in the analysis_results.csv table.
visualize_experiment_full(experiment_id=34)

## 5. Core Engine: Adaptive Online ARIMA Benchmark

This function implements the primary competitor for DriftMind in this notebook: an **Online Adaptive ARIMA** loop. Unlike a static model, this engine monitors its own health in real-time and "self-corrects" when it detects that the underlying data distribution has shifted.

#### Adaptive Mechanism:
* **Volatility-Aware Scaling**: It uses the initial training data to establish a "Naive Error" baseline, allowing for the calculation of the **Mean Absolute Scaled Error (MASE)**—a metric that is independent of the data's absolute magnitude.
* **Dual-Trigger Logic**:
    * **Accuracy Trigger (MASE)**: If the error exceeds the `mase_limit` for a consecutive number of steps, the model assumes a "Mean Shift" drift has occurred.
    * **Structural Trigger (Correlation)**: If the correlation between predictions and actuals drops below the `corr_floor`, it assumes a "Structural" drift has occurred (the pattern changed, even if the error is still low).
* **Self-Healing**: When a trigger fires, the model discards its old parameters and performs a full re-fit on the most recent data window to adapt to the new regime.

In [ ]:
def adaptive_online_arima_benchmark(
    df,
    training_fraction=0.01,
    arima_order=(5, 1, 0),
    arima_fit_window=200,
    mase_limit=3.0,
    mase_window=10,
    mase_consecutive_steps=20,
    corr_floor=0.20,
    eval_window=100,
):
    y = df["val"].to_numpy()
    n = len(y)
    split_idx = int(n * training_fraction)

    # Initial Training
    train_data = y[:split_idx]
    history = list(train_data)

    # --- Initialize Adaptive Metrics ---
    # Scaling factor for MASE (Volatility)
    naive_scaling_factor = max(np.nanmean(np.abs(np.diff(train_data))), 1e-8)

    model = TriggeredARIMABaseline(order=arima_order)
    model.train(train_data[-arima_fit_window:])

    results = []
    recent_abs_errors = []
    recent_preds = []
    recent_actuals = []

    mase_bad_run = 0
    total_retrains = 0
    total_mae = 0
    inference_count = 0

    print("Executing Adaptive Online ARIMA...")
    test_data = y[split_idx:]

    for i, actual_val in enumerate(tqdm(test_data)):
        global_pos = split_idx + i
        window = history[-arima_fit_window:]

        # 1. Predict with current frozen parameters
        pred, latency = model.predict_point(window)

        # 2. Compute immediate error
        ae = abs(actual_val - pred) if np.isfinite(pred) else 1e6
        recent_abs_errors.append(ae)
        recent_preds.append(pred)
        recent_actuals.append(actual_val)

        # Keep evaluation windows lean
        if len(recent_abs_errors) > eval_window:
            recent_abs_errors.pop(0)
            recent_preds.pop(0)
            recent_actuals.pop(0)

        # 3. Calculate Adaptive Metrics
        current_mae = np.mean(recent_abs_errors[-mase_window:])
        mase = current_mae / naive_scaling_factor

        # Correlation check
        corr = 1.0
        if len(recent_preds) >= eval_window:
            corr = np.corrcoef(recent_preds, recent_actuals)[0, 1]

        # 4. TRIGGER LOGIC
        trigger_fired = False

        if mase > mase_limit:
            mase_bad_run += 1
            if mase_bad_run >= mase_consecutive_steps:
                trigger_fired = True
        elif corr < corr_floor:
            trigger_fired = True
        else:
            mase_bad_run = 0

        # 5. ADAPTIVE RESET
        if trigger_fired:
            new_train_window = history[-arima_fit_window:]
            # Update Scaling (Naive Error)
            naive_scaling_factor = max(
                np.nanmean(np.abs(np.diff(new_train_window))), 1e-8
            )
            # Retrain Model
            model.train(new_train_window)
            mase_bad_run = 0
            total_retrains += 1

        total_mae = total_mae + abs(actual_val - pred)
        inference_count += 1
        results.append(
            {
                "pos": global_pos,
                "expected": actual_val,
                "predicted": pred,
                "mase": mase,
                "corr": corr,
                "latency": latency,
                "retrain": 1 if trigger_fired else 0,
            }
        )

        history.append(actual_val)

    print(f"Retrains: {total_retrains}")
    total_mae = total_mae / inference_count
    print(f"Avg MAE: {total_mae}")
    return pd.DataFrame(results)

## 6. Head-to-Head Benchmark: Orchestration Function

`benchmark_with_target` is the top-level orchestrator that ties the full comparison pipeline together for a given experiment.

#### Workflow:
1. **Metadata Lookup**: Loads `analysis_results.csv` to retrieve DriftMind's pre-computed performance metrics (MAE, sMAE, Throughput) for the target experiment.
2. **Data Loading**: Locates the raw prediction log for the experiment via glob and reads it into a DataFrame.
3. **ARIMA Execution**: Extracts the ground-truth signal from the DriftMind results file and passes it to `adaptive_online_arima_benchmark` for a live run, timing it with `time.perf_counter()` for high-resolution measurement.
4. **Metric Comparison**: Computes ARIMA's MAE, sMAE, and throughput, then prints a side-by-side summary table.
5. **Dual-Panel Visualization**: Renders a two-pane chart — prediction overlap (top) and absolute-error comparison (bottom) — annotated with accuracy and speed winner labels.

In [ ]:
def benchmark_with_target(
    experiment_id,
    dm_results_folder=None,
    main_results_csv=None,
):
    """
    Retrieves original signal, runs ARIMA, and generates a two-part
    comparison: Predictions and Absolute Error residuals.

    Args:
        experiment_id (int): The unique ID assigned to the data run.
        dm_results_folder (Path | None): Directory containing raw DriftMind prediction logs.
            Defaults to EXPERIMENTS_DIR.
        main_results_csv (Path | None): Path to the summarized results table.
            Defaults to ANALYSIS_CSV.
    """
    if dm_results_folder is None:
        dm_results_folder = EXPERIMENTS_DIR
    if main_results_csv is None:
        main_results_csv = ANALYSIS_CSV

    # 1. Load Metadata
    df_main = pd.read_csv(main_results_csv)
    if "Experiment Id" not in df_main.columns:
        cols = [
            "Experiment Id",
            "Category",
            "Dataset",
            "Mean",
            "StdDev",
            "Min",
            "Max",
            "Range",
            "MAE",
            "sMAE",
            "Time",
            "Points",
            "Throughput",
        ]
        df_main = pd.read_csv(main_results_csv, names=cols, header=None)

    exp_info = df_main[df_main["Experiment Id"] == experiment_id]
    if exp_info.empty:
        print(f"Error: Experiment ID {experiment_id} not found.")
        return

    meta = exp_info.iloc[0]

    # 2. Load DriftMind Results
    matches = list(dm_results_folder.glob(f"Experiment_{experiment_id}_*.csv"))
    if not matches:
        print(f"Error: File for Exp {experiment_id} not found.")
        return

    dm_file = matches[0]
    df_dm = pd.read_csv(dm_file, skipinitialspace=True)

    # 3. Run ARIMA Benchmark
    df_for_arima = pd.DataFrame({"val": df_dm["Actual"]})
    print(f"Executing ARIMA benchmark for: {dm_file.name}")

    start_time = time.perf_counter()
    df_arima = adaptive_online_arima_benchmark(df_for_arima)
    arima_duration = time.perf_counter() - start_time

    # Calculate Metrics
    arima_ae = np.abs(df_arima["expected"] - df_arima["predicted"])
    arima_mae = np.mean(arima_ae)
    arima_smae = arima_mae / meta["Range"]
    arima_tps = len(df_dm) / arima_duration if arima_duration > 0 else 0

    # 4. Print Summary
    print("\n" + "=" * 60)
    print(" COMPARISON: DRIFTMIND VS. ADAPTIVE ARIMA")
    print("=" * 60)
    print(f"{'Metric':<15} | {'DriftMind':<15} | {'ARIMA':<15}")
    print("-" * 50)
    print(f"{'MAE':<15} | {meta['MAE']:<15.4f} | {arima_mae:<15.4f}")
    print(f"{'sMAE':<15} | {meta['sMAE']:<15.4f} | {arima_smae:<15.4f}")
    print(f"{'Throughput':<15} | {meta['Throughput']:<15.2f} | {arima_tps:<15.2f}")
    print("=" * 60)

    # 5. Visualization (Dual Subplots)
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(16, 12), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
    )

    # --- Top Plot: Predictions ---
    ax1.plot(
        df_dm.index,
        df_dm["Actual"],
        label="Actual (Ground Truth)",
        color="black",
        alpha=0.2,
        linewidth=1,
    )
    ax1.plot(
        df_dm.index,
        df_dm["Expected"],
        label="DriftMind Prediction",
        color="#1f77b4",
        linewidth=2,
    )
    ax1.plot(
        df_arima["pos"],
        df_arima["predicted"],
        label="Adaptive ARIMA",
        color="#d62728",
        linestyle="--",
        alpha=0.8,
    )

    ax1.set_title(f"Model Comparison: {dm_file.name}", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Value")
    ax1.legend(loc="best", shadow=True)
    ax1.grid(True, alpha=0.2)

    # --- Bottom Plot: Absolute Error (AE) Comparison ---
    # Plotting DriftMind AE
    ax2.plot(df_dm.index, df_dm["AE"], label="DriftMind AE", color="#1f77b4", alpha=0.7)
    ax2.fill_between(df_dm.index, df_dm["AE"], color="#1f77b4", alpha=0.1)

    # Plotting ARIMA AE
    ax2.plot(df_arima["pos"], arima_ae, label="ARIMA AE", color="#d62728", alpha=0.7)
    ax2.fill_between(df_arima["pos"], arima_ae, color="#d62728", alpha=0.1)

    ax2.set_title("Absolute Error (AE) Comparison", fontsize=12)
    ax2.set_ylabel("Error Amplitude")
    ax2.set_xlabel("Time Step")
    ax2.legend(loc="best", shadow=True)
    ax2.grid(True, alpha=0.2)

    # Winner Labels
    # 1. Accuracy Winner (Normalized to Signal Range)
    # This represents how much of the "Total Signal Scale" the winner saved you
    acc_diff_absolute = abs(arima_mae - meta["MAE"])
    acc_improvement_pct = (acc_diff_absolute / meta["Range"]) * 100

    acc_winner = "DriftMind" if meta["MAE"] < arima_mae else "ARIMA"
    acc_color = "green" if acc_winner == "DriftMind" else "red"
    acc_text = (
        f"ACCURACY WINNER: {acc_winner} ({acc_improvement_pct:.2f}% of Range better)"
    )

    # 2. Throughput Winner (Keep as % faster, as speed is relative)
    tps_denom = arima_tps if arima_tps > 0 else 1
    tps_diff = ((meta["Throughput"] - arima_tps) / tps_denom) * 100
    tps_winner = "DriftMind" if meta["Throughput"] > arima_tps else "ARIMA"
    tps_color = "green" if tps_winner == "DriftMind" else "red"
    tps_text = f"SPEED WINNER: {tps_winner} ({abs(tps_diff):.1f}% faster)"

    # Add labels to the top plot (ax1)
    ax1.text(
        0.01,
        0.94,
        acc_text,
        transform=ax1.transAxes,
        fontsize=11,
        fontweight="bold",
        color=acc_color,
        bbox=dict(facecolor="white", alpha=0.9, edgecolor="black"),
    )
    ax1.text(
        0.01,
        0.88,
        tps_text,
        transform=ax1.transAxes,
        fontsize=11,
        fontweight="bold",
        color=tps_color,
        bbox=dict(facecolor="white", alpha=0.9, edgecolor="black"),
    )

    plt.tight_layout()
    plt.show()

## 7. Execution: Head-to-Head Comparison

Run `benchmark_with_target` for a specific experiment. Change `experiment_id` to any value present in `analysis_results.csv`.

In [ ]:
# --- EXECUTION: HEAD-TO-HEAD BATTLE ---
# We execute the full comparison for Experiment 1.
# This single call orchestrates data loading, ARIMA adaptation,
# metric calculation, and visualization generation.
benchmark_with_target(experiment_id=34)